# PolySurrogate — Sample Workflow

This notebook walks through the **complete pipeline**: three self-contained
scripts at the project root, run in sequence, sharing only `utils.py` for
genuinely common code (PyTheus catalog/amplitude computation, model
definitions, reproducibility primitives).

| Stage | Script | What happens |
|-------|--------|--------------|
| 1 | `01_data_generate.py` | random graph weights -> unnormalised quantum amplitude vectors, saved + merged |
| 2 | `02_ml_model.py` | train a surrogate PNN (Polynomial Neural Network) |
| 3 | `inverse_design.py` | optimise graph weights toward a GHZ target state |
| 4 | -- | output inspection |

Stage 2 reads the `.npz` file stage 1 wrote; stage 3 reads the
`params.msgpack` stage 2 wrote -- they connect only through those files,
never through a Python import between the numbered scripts themselves.

### This is a toy / demo run

All settings are deliberately tiny so the notebook completes in **under two
minutes on a laptop CPU**. Model accuracy and optimisation fidelity will be
low -- that is expected and intentional; the goal is to show the pipeline
runs end-to-end with every parameter visible and editable below.

---
## Setup

Loads the three numbered scripts with `importlib.import_module` (the repo
root is added to `sys.path` first) and creates the demo output tree under
`results/sample_workflow/`.

Note: a literal `import 01_data_generate` is not valid Python -- identifiers
can't start with a digit -- but `importlib.import_module("01_data_generate")`
works fine since it takes the module name as a string.

**No edits needed here.**

In [1]:
import sys
import json
import importlib
import time
from pathlib import Path

# ---------------------------------------------------------------------------
# Auto-detect project root: works whether launched from the project root
# or from inside notebooks/
# ---------------------------------------------------------------------------
_cwd = Path.cwd()
if (_cwd / "01_data_generate.py").exists():
    REPO_ROOT = _cwd
elif (_cwd.parent / "01_data_generate.py").exists():
    REPO_ROOT = _cwd.parent
else:
    raise RuntimeError(
        f"Cannot locate the project root from {_cwd}.\n"
        "Run this notebook from the project root or from the notebooks/ folder."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

stage1 = importlib.import_module("01_data_generate")
stage2 = importlib.import_module("02_ml_model")
stage3 = importlib.import_module("inverse_design")

# ---------------------------------------------------------------------------
# All demo outputs live under results/sample_workflow/ -- the exact
# subfolders (which need NODES) are derived at the end of the Parameters
# cell below, once NODES is defined.
# ---------------------------------------------------------------------------
RESULTS_ROOT = REPO_ROOT / "results" / "sample_workflow"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

import numpy as np
import jax

print("Project root :", REPO_ROOT)
print("Results root :", RESULTS_ROOT)
print()
print("JAX devices  :", jax.devices())
print("JAX backend  :", jax.default_backend())

Project root : /home/bo48god/quick_tests/PolySurrogate
Results root : /home/bo48god/quick_tests/PolySurrogate/results/sample_workflow



ERROR:jax._src.xla_bridge:Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/home/bo48god/.conda/envs/env_jax/lib/python3.11/site-packages/jax/_src/xla_bridge.py", line 487, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/home/bo48god/.conda/envs/env_jax/lib/python3.11/site-packages/jax_plugins/xla_cuda12/__init__.py", line 370, in initialize
    _check_cuda_versions(raise_on_first_error = True)
  File "/home/bo48god/.conda/envs/env_jax/lib/python3.11/site-packages/jax_plugins/xla_cuda12/__init__.py", line 274, in _check_cuda_versions
    for d in range(cuda_versions.cuda_device_count())
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: jaxlib/cuda/versions_helpers.cc:134: operation cuInit(0) failed: Unknown CUDA error 303; cuGetErrorName failed. This probably means that JAX was unable to load the CUDA libraries.


JAX devices  : [CpuDevice(id=0)]
JAX backend  : cpu


---
## Parameters

**Minimal parameters** -- the handful of things you're most likely to want
to change for a quick experiment. Everything else (learning rate, split
ratios, jitter schedule, pruning thresholds, ...) uses the same sensible
defaults as the standalone scripts. For the full parameter reference across
all three stages, see [`PARAMETERS.md`](../PARAMETERS.md) in the project root.

In [2]:
# =============================================================================
# Minimal parameters -- edit these to experiment. Full reference: PARAMETERS.md
# =============================================================================

# ---- System size (Stage 1/2/3 -- must match everywhere) ----
NODES      = 4       # graph vertices / photons / modes
DIMENSIONS = 2       # local Hilbert-space dimension (2 = qubit-like)

# ---- Stage 1: Data generation ----
N_SAMPLES = 500      # total samples to generate for training
DATA_SEED = 42       # reproducibility seed for data generation

# ---- Stage 2: Model training ----
MODEL_TYPE = "PNN"   # "PNN" (one hidden layer, monomial activation) or "FNN" (multi-layer, GELU)
HIDDEN_DIM = 400     # PNN: int hidden dim. FNN: tuple, e.g. (400, 400)
NUM_EPOCHS = 15      # training epochs
TRAIN_SEED = 42      # master seed: model init, data split, epoch shuffling

# ---- Stage 3: Inverse-design optimisation ----
TARGET      = "GHZ"  # "GHZ" | "W" | "LINEAR_CLUSTER" | "SINGLE" | "ZERO"
OPT_SAMPLES = 3       # starting samples to optimise
OPT_STEPS   = 200     # max optimisation steps per sample
OPT_SEED    = 46      # master seed: drives all jitter noise

# =============================================================================
# Derived output paths.
#
# DATA_ROOT is just the parent for data generation -- 01_data_generate.py
# auto-numbers the actual n{NODES}/n{NODES}_{i} folder itself (see the Stage
# 1 cell below), so nothing here ever names or pre-creates that folder.
# =============================================================================

DATA_ROOT = RESULTS_ROOT / "data_generation"
MODEL_DIR = RESULTS_ROOT / "model_training"
OPT_DIR   = RESULTS_ROOT / "inverse_design"

for _d in [MODEL_DIR, OPT_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

---
## Stage 1 — Data generation

`01_data_generate.py`'s `generate_dataset()`:
1. Builds a PyTheus perfect-matching catalogue for an `n`-node graph.
2. Draws random graph weight vectors uniformly from `[-1, 1]`.
3. Computes the **unnormalised** quantum amplitude vector for each weight sample.
4. Saves shards and merges them into one `dataset_merged.npz` file.

**Input shape:** `(N_SAMPLES, INPUT_DIM)` -- one weight vector per sample
**Output shape:** `(N_SAMPLES, OUT_DIM)` -- one unnormalised amplitude vector per sample

In [3]:
print(f"Generating {N_SAMPLES} samples for {NODES}-node graph...")
t0 = time.perf_counter()

# save_data=True -> written to disk at an auto-numbered n{NODES}/n{NODES}_{i}
# folder under DATA_ROOT (never a name we choose here); always returns the
# in-memory arrays too, regardless of save_data. batch_size/shard_size/
# normed_data use the same defaults as 01_data_generate.py -- see
# PARAMETERS.md to change them.
weights, amps, data_out_dir = stage1.generate_dataset(
    vertices     = NODES,
    dimensions   = DIMENSIONS,
    n_samples    = N_SAMPLES,
    batch_size   = 100,
    seed         = DATA_SEED,
    shard_size   = N_SAMPLES,
    normed_data  = False,
    save_data    = True,
    data_root    = str(DATA_ROOT),
)

elapsed = time.perf_counter() - t0
merged_path = data_out_dir / "dataset_merged.npz"
print(f"\nDone in {elapsed:.1f}s")
print(f"weights={weights.shape}  amps={amps.shape}")
print(f"Merged dataset: {merged_path}")

MERGED_PATH = str(merged_path)

Generating 500 samples for 4-node graph...
2026-07-21 15:45:32,697 | INFO | Log file: /home/bo48god/quick_tests/PolySurrogate/results/sample_workflow/data_generation/n4/n4_4/logs/data_generation_20260721_154532.log


2026-07-21 15:45:32,698 | INFO | === Random Dense Graph Dataset Generation ===


2026-07-21 15:45:32,699 | INFO | JAX devices: [CpuDevice(id=0)]


2026-07-21 15:45:32,700 | INFO | vertices=4, dimensions=2


2026-07-21 15:45:32,701 | INFO | n_samples=500, batch_size=100


...

2026-07-21 15:45:33,000 | INFO | === DONE === total time=0.0s


2026-07-21 15:45:33,001 | INFO | Merging shards...


2026-07-21 15:45:33,043 | INFO | Merged dataset -> /home/bo48god/quick_tests/PolySurrogate/results/sample_workflow/data_generation/n4/n4_4/dataset_merged.npz


2026-07-21 15:45:33,240 | INFO | Wrote reproducibility_manifest.json -> /home/bo48god/quick_tests/PolySurrogate/results/sample_workflow/data_generation/n4/n4_4/reproducibility_manifest.json



Done in 0.6s
weights=(500, 24)  amps=(500, 16)
Merged dataset: /home/bo48god/quick_tests/PolySurrogate/results/sample_workflow/data_generation/n4/n4_4/dataset_merged.npz


### Peek at one data sample

Each row of `weights` is a flat vector of graph edge weights.
Each row of `amps` is the corresponding quantum amplitude vector.

In [4]:
d = np.load(MERGED_PATH)
w0, a0 = d["weights"][0], d["amps"][0]

print("Sample 0 -- graph weights (first 8 of", len(w0), "edges):")
print(" ", np.round(w0, 4))
print()
print("Sample 0 -- amplitude vector (all", len(a0), "components):")
print(" ", np.round(a0, 4))
print()
print("Amplitude norm:", round(float(np.linalg.norm(a0)), 6),
      "(is 1.0 only when NORMED_DATA=True; unnormalised otherwise)")

Sample 0 -- graph weights (first 8 of 24 edges):
  [ 0.5479 -0.1222  0.7172  0.3947 -0.8116  0.9512  0.5223  0.5721 -0.7438
 -0.0992 -0.2584  0.8535  0.2877  0.6455 -0.1132 -0.5455  0.1092 -0.8724
  0.6553  0.2633  0.5162 -0.2909  0.9414  0.7862]

Sample 0 -- amplitude vector (all 16 components):
  [-0.0198  0.5201  0.1395 -0.4631 -0.5108 -0.1669  0.914   0.2085  0.3529
 -0.4187  0.5708  0.6158  0.5752 -0.0739  0.8875 -0.0046]

Amplitude norm: 1.949658 (is 1.0 only when NORMED_DATA=True; unnormalised otherwise)


---
## Stage 2 — Model training

`02_ml_model.py`'s `train_surrogate_model(cfg)` trains the surrogate model
to map graph weights to **unnormalised** amplitude vectors, minimising MAE.

- `PNN` (Polynomial Neural Network): one hidden layer of size `HIDDEN_DIM` with a monomial activation `x^(n/2)`.
- `FNN`: a standard multi-layer network (pass a tuple for `HIDDEN_DIM`).

A `cfg` dict (built from the Parameters cell above) is passed directly to
`train_surrogate_model()` -- no config file edit needed, and no
`importlib.reload` dance either (this function takes `cfg` as a real
parameter).

> **Low fidelity is expected for this tiny demo.** With only 15 training
> epochs, the surrogate has not converged. The purpose here is to show that
> the pipeline runs end-to-end.

In [5]:
# Everything except NODES/MODEL_TYPE/HIDDEN_DIM/NUM_EPOCHS/TRAIN_SEED uses
# the same defaults as 02_ml_model.py's own top-of-file constants -- see
# PARAMETERS.md to change any of them.
TRAINING_CFG = dict(
    NODES=NODES, DIMENSIONS=DIMENSIONS, DATE="sample_workflow", MODEL_NAME=MODEL_TYPE,
    HIDDEN_DIM=HIDDEN_DIM, DATA_PATH=MERGED_PATH, DATA_SIZE=None,
    NORMALIZE_MODEL_OUTPUT=False,
    TRAIN_SPLIT=0.8, VAL_SPLIT=0.1, TEST_SPLIT=0.1,
    LEARNING_RATE=1e-3, LR_AFTER_DECAY=1e-5, LR_DECAY_UNTIL_EPOCH=NUM_EPOCHS,
    BATCH_SIZE=50, NUM_EPOCHS=NUM_EPOCHS, PATIENCE=5, TOLERANCE=1e-7,
    SEED=TRAIN_SEED, SPLIT_MODE="contiguous", PRECISION="float32",
    DATASET_HASH_MODE="fast", DEBUG_TRACE=False,
    RESUME_FULL_STATE=False, CKPT_DIR_RESTORE="",
    ROOT_FOLDER=str(MODEL_DIR), LOSS_NAME="mae",
    DETERMINISTIC_XLA=True,
)

print(f"Training {MODEL_TYPE} (HIDDEN_DIM={HIDDEN_DIM}, NODES={NODES}) for {NUM_EPOCHS} epochs...")
t0 = time.perf_counter()

run_dir = stage2.train_surrogate_model(TRAINING_CFG)
run_dir = Path(run_dir)

elapsed = time.perf_counter() - t0
print(f"\nTraining complete in {elapsed:.1f}s")
print(f"Run directory: {run_dir}")
print("Output files:")
for f in sorted(run_dir.glob("*")):
    print(f"  {f.name}")

Training PNN (HIDDEN_DIM=400, NODES=4) for 15 epochs...



Training complete in 7.5s
Run directory: /home/bo48god/quick_tests/PolySurrogate/results/sample_workflow/model_training/PNN/n4/n4_4
Output files:
  checkpoints
  model_info.json
  params.msgpack
  params_snapshots
  plots
  reproducibility_manifest.json
  run_log.txt
  test_metrics.npz


In [6]:
# Confirm params.msgpack was saved and read back some training info
params_path = run_dir / "params.msgpack"
model_info_path = run_dir / "model_info.json"

assert params_path.exists(), "params.msgpack not found -- training may have failed"
print(f"params.msgpack : {params_path.stat().st_size:,} bytes")

with open(model_info_path) as f:
    info = json.load(f)

keys_to_show = ["model_name", "architecture", "input_dims", "output_dims",
                "num_epoch", "batch_size", "best_val_loss", "best_epoch"]
print("\nTraining summary:")
for k in keys_to_show:
    if k in info:
        print(f"  {k}: {info[k]}")

PARAMS_PATH = str(params_path)

params.msgpack : 65,793 bytes

Training summary:
  model_name: PNN
  architecture: 400
  input_dims: 24
  output_dims: 16
  num_epoch: 15
  batch_size: 50
  best_val_loss: 0.5777239799499512
  best_epoch: 14


---
## Stage 3 — Inverse-design optimisation

The trained surrogate is used as a differentiable proxy for PyTheus.
Gradient descent on the graph weights minimises:

$$\mathcal{L}(x) = (1 - \text{fidelity}(x)) + \lambda_{L1} \|x\|_1$$

The $L_1$ penalty encourages a sparse solution (few active graph edges).
Stopping condition is a single threshold: `nn_fid >= EARLY_STOP_NN_FID OR
step >= OPT_STEPS`. If fidelity stalls, reproducible jitter (escalating
random re-perturbation) kicks in -- every jitter draw is a pure function of
`(OPT_SEED, sample_id, event_index)`, so a sample's noise sequence is
independent of run order or which subset it's part of. After the loop, the
graph is **pruned** by zeroing small weights while keeping fidelity within
tolerance.

> **Low fidelity is expected for this tiny demo.** With only 15 training
> epochs, the surrogate has not converged, so this is testing that the
> *pipeline* runs end-to-end, not real inverse-design quality.

The `cfg` dict below uses TARGET/OPT_SAMPLES/OPT_STEPS/OPT_SEED from the
Parameters cell above; everything else uses the same defaults as
`inverse_design.py` -- see PARAMETERS.md for the full reference.

In [7]:
# Everything except n/target_name/model_type/data_samples/num_steps/seed
# uses the same defaults as inverse_design.py's own top-of-file constants
# -- see PARAMETERS.md to change any of them.
OPTIMISER_CFG = dict(
    # ------------------------------------------------------------------
    # Problem setup
    # ------------------------------------------------------------------
    n=NODES, dimensions=DIMENSIONS, target_name=TARGET, model_type=MODEL_TYPE,

    # ------------------------------------------------------------------
    # Starting samples
    # ------------------------------------------------------------------
    generate_data=True, conditioned_data_path="", max_initial_samples=None,
    data_samples=OPT_SAMPLES, data_batch_size=OPT_SAMPLES, data_seed=7,
    low_fidelity_threshold=0.999,
    normed_data=False,  # must match normed_data used in data generation
    generation_gpu_batch_size=OPT_SAMPLES, data_shard_size=OPT_SAMPLES,

    # ------------------------------------------------------------------
    # Trained model
    # ------------------------------------------------------------------
    architecture=HIDDEN_DIM,  # must match HIDDEN_DIM from training
    model_path=PARAMS_PATH,
    normalize_model_output=False,
    input_dim=2 * NODES * (NODES - 1), out_dim=2 ** NODES,

    # ------------------------------------------------------------------
    # Objective and schedule
    # ------------------------------------------------------------------
    lambda_l1=1e-3, seed=OPT_SEED, num_steps=OPT_STEPS,
    early_stop_nn_fid=0.99999,
    learning_rate=1e-2, min_learning_rate=1e-6,
    lr_decay_steps=OPT_STEPS, lr_exponent=1.0,
    clip_min=-1.0, clip_max=1.0,

    # ------------------------------------------------------------------
    # Jittering (see markdown cell above)
    # ------------------------------------------------------------------
    jitter_enabled=True, initial_jitter=0.01,
    jitter_schedule=[0.01, 0.05, 0.10, 0.20, 0.30, 0.50, 0.70, 1.00],
    jitter_max_events=8, stuck_patience_steps=50, stuck_min_improvement=1e-4,
    max_total_steps=OPT_STEPS,

    # ------------------------------------------------------------------
    # Logging
    # ------------------------------------------------------------------
    print_every=20, verify_every=20, store_step_vectors=False,

    # ------------------------------------------------------------------
    # Pruning
    # ------------------------------------------------------------------
    prune_fid_tolerance=1e-4, prune_thresholds=[1e-5, 1e-4, 1e-3, 1e-2, 1e-1],

    # ------------------------------------------------------------------
    # Output
    # ------------------------------------------------------------------
    results_root=str(OPT_DIR),
)

print(f"Optimising {OPT_SAMPLES} samples toward {TARGET} (n={NODES}) for up to {OPT_STEPS} steps each...")
t0 = time.perf_counter()

result_dir = stage3.run_optimisation(OPTIMISER_CFG)
result_dir = Path(result_dir)

elapsed = time.perf_counter() - t0
print(f"\nOptimisation complete in {elapsed:.1f}s")
print(f"Result directory: {result_dir}")

Optimising 3 samples toward GHZ (n=4) for up to 200 steps each...
2026-07-21 15:45:40,760 | INFO | === Random Dense Graph Dataset Generation ===


2026-07-21 15:45:40,762 | INFO | JAX devices: [CpuDevice(id=0)]


2026-07-21 15:45:40,762 | INFO | vertices=4, dimensions=2


2026-07-21 15:45:40,763 | INFO | n_samples=3, batch_size=3


2026-07-21 15:45:40,765 | INFO | seed=7


...

2026-07-21 15:45:40,906 | INFO | Generating data batches...


2026-07-21 15:45:40,927 | INFO | batch     1 |          3/3 (100.0%) | 0.0s | 155 samples/s


2026-07-21 15:45:40,928 | INFO | === DONE === total time=0.0s


round=1 | generated=3 | kept=3 | discarded=0 | total_kept=3/3

Final low-fidelity dataset check:
total generated = 3
final dataset size = 3
max fidelity = 0.06815741211175919
all fidelities < threshold = True



Optimisation complete in 6.2s
Result directory: /home/bo48god/quick_tests/PolySurrogate/results/sample_workflow/inverse_design/GHZ_n4/GHZ_n4_3


---
## Stage 4 — Load and inspect outputs

Everything the notebook produced lives under `results/sample_workflow/`.
This section lists the files and prints a short summary.

In [8]:
print("=" * 60)
print(f"All files under {RESULTS_ROOT.relative_to(REPO_ROOT)}/")
print("=" * 60)

all_files = sorted(RESULTS_ROOT.rglob("*"))
for p in all_files:
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        rel = p.relative_to(RESULTS_ROOT)
        print(f"  {str(rel):<60s}  {size_kb:6.1f} KB")

All files under results/sample_workflow/


  data_generation/n4/n4_0/data_00000.npz                          78.6 KB
  data_generation/n4/n4_0/dataset_merged.npz                      78.6 KB
  data_generation/n4/n4_0/logs/data_generation_20260721_141402.log     1.6 KB
  data_generation/n4/n4_0/merged_amps.dat                         31.2 KB
  data_generation/n4/n4_0/merged_weights.dat                      46.9 KB
  data_generation/n4/n4_0/metadata.json                            0.3 KB
  data_generation/n4/n4_0/reproducibility_manifest.json            1.6 KB
  data_generation/n4/n4_1/data_00000.npz                          78.6 KB
  data_generation/n4/n4_1/dataset_merged.npz                      78.6 KB
  data_generation/n4/n4_1/logs/data_generation_20260721_151221.log     1.6 KB
  data_generation/n4/n4_1/merged_amps.dat                         31.2 KB
  data_generation/n4/n4_1/merged_weights.dat                      46.9 KB
  data_generation/n4/n4_1/metadata.json                            0.3 KB
  data_generation/n4/n4_1/repr

  model_training/PNN/n4/n4_1/params_snapshots/best_param_epoch_1.msgpack    64.3 KB
  model_training/PNN/n4/n4_1/params_snapshots/best_param_epoch_10.msgpack    64.3 KB
  model_training/PNN/n4/n4_1/params_snapshots/best_param_epoch_11.msgpack    64.3 KB
  model_training/PNN/n4/n4_1/params_snapshots/best_param_epoch_12.msgpack    64.3 KB
  model_training/PNN/n4/n4_1/params_snapshots/best_param_epoch_13.msgpack    64.3 KB
  model_training/PNN/n4/n4_1/params_snapshots/best_param_epoch_14.msgpack    64.3 KB
  model_training/PNN/n4/n4_1/params_snapshots/best_param_epoch_2.msgpack    64.3 KB
  model_training/PNN/n4/n4_1/params_snapshots/best_param_epoch_3.msgpack    64.3 KB
  model_training/PNN/n4/n4_1/params_snapshots/best_param_epoch_4.msgpack    64.3 KB
  model_training/PNN/n4/n4_1/params_snapshots/best_param_epoch_5.msgpack    64.3 KB
  model_training/PNN/n4/n4_1/params_snapshots/best_param_epoch_6.msgpack    64.3 KB
  model_training/PNN/n4/n4_1/params_snapshots/best_param_epoch_7.msgpac

In [9]:
# Read the optimisation summary
summary_file = result_dir / "optimisation_summary.json"
with open(summary_file) as f:
    summary = json.load(f)

print("Optimisation summary")
print("-" * 40)
print(f"Samples run             : {len(summary['final_pytheus_fidelities'])}")
print(f"Steps per sample        : {summary['optimisation_steps']}")
print(f"Jitter events per sample: {summary['jitter_counts']}")
print()
print("PyTheus fidelity to", TARGET, "before pruning (higher = better, max = 1.0):")
for i, fid in enumerate(summary["pytheus_fidelity_before_pruning"]):
    print(f"  sample {i}: {fid:.4f}")
print()
best_fid = max(summary["final_pytheus_fidelities"])
print(f"Best fidelity after pruning in demo: {best_fid:.4f}")
print()
print("Note: low fidelity is expected for this tiny demo run.")
print("For meaningful results, increase N_SAMPLES, NUM_EPOCHS, and OPT_STEPS.")

Optimisation summary
----------------------------------------
Samples run             : 3
Steps per sample        : [200, 200, 200]
Jitter events per sample: [0, 0, 0]

PyTheus fidelity to GHZ before pruning (higher = better, max = 1.0):
  sample 0: 0.1253
  sample 1: 0.0447
  sample 2: 0.0713

Best fidelity after pruning in demo: 0.1255

Note: low fidelity is expected for this tiny demo run.
For meaningful results, increase N_SAMPLES, NUM_EPOCHS, and OPT_STEPS.


In [10]:
# Print the best-graph solution found
best_file = result_dir / "best_graph_solution.json"
with open(best_file) as f:
    best = json.load(f)

print("Best graph found:")
print(f"  Sample ID       : {best['sample_id']}")
print(f"  Active edges    : {best['num_edges']}  (non-zero after pruning)")
print(f"  PyTheus fidelity: {best['fidelity']:.4f}")
print()
print(f"Graph weight vector ({2 * NODES * (NODES - 1)} edges for n={NODES}):")
gv = best['graph_vector']
active = [(i, round(v, 4)) for i, v in enumerate(gv) if abs(v) > 1e-4]
print(f"  {len(active)} non-zero weights:")
for idx, val in active:
    print(f"    edge {idx:2d}: {val:+.4f}")

Best graph found:
  Sample ID       : 0
  Active edges    : 22  (non-zero after pruning)
  PyTheus fidelity: 0.1255

Graph weight vector (24 edges for n=4):
  22 non-zero weights:
    edge  0: +0.6083
    edge  1: +0.7704
    edge  2: +0.7428
    edge  3: +0.1295
    edge  4: -0.0475
    edge  5: +1.0000
    edge  7: -0.0287
    edge  8: -0.4478
    edge  9: +0.5492
    edge 10: +0.0530
    edge 11: -0.5244
    edge 13: -0.5495
    edge 14: -0.1308
    edge 15: +0.8102
    edge 16: +0.5080
    edge 17: +0.6699
    edge 18: +0.5929
    edge 19: +0.1301
    edge 20: -0.7392
    edge 21: -1.0000
    edge 22: -0.7467
    edge 23: -0.2593


### Reproducibility check

Every stage writes a `reproducibility_manifest.json` recording the git
commit, environment (JAX/jaxlib/flax/optax versions, devices), every seed
used, and a hash of the run's key input file (dataset for training, model
checkpoint for inverse design) -- enough to know exactly what produced a
given run's output later, and to rerun it identically.

In [11]:
manifest_file = result_dir / "reproducibility_manifest.json"
with open(manifest_file) as f:
    manifest = json.load(f)

print("Stage 3 reproducibility manifest")
print("-" * 40)
print("git:", manifest["git"])
print("seeds:", manifest["seeds"])
print("jax_devices:", manifest["environment"]["jax_devices"])
print("model checkpoint hash present:", "sha256_fast" in manifest["input_file"])

with open(run_dir / "reproducibility_manifest.json") as f:
    train_manifest = json.load(f)
print()
print("Stage 2 reproducibility manifest")
print("-" * 40)
print("seeds:", train_manifest["seeds"])
print("dataset hash present:", "sha256_fast" in train_manifest["input_file"])

Stage 3 reproducibility manifest
----------------------------------------
git: {'available': True, 'commit': 'd5ac9eb86d56f495622a2cc93cef0c15b3c92578', 'dirty': True, 'branch': 'main'}
seeds: {'seed': 46, 'data_seed': 7}
jax_devices: ['TFRT_CPU_0']
model checkpoint hash present: True

Stage 2 reproducibility manifest
----------------------------------------
seeds: {'seed': 42, 'key_model_init': '48ef9e441264be9a5325f40b72ed99c6e10dee6de9af26fb24a23ec09f3c237d', 'key_data_split': '4ee89e22abd833d3ab09b2d5c6dc5d8a1fd189fb698280c1fad64b4d513680fc', 'key_epoch_root': '55e0a4f3e356c611bd3d638ba4a23fccb0e49afb2616568539f210bd3ac68722', 'key_batch_root': '0bb78ced70e14f9e3d693015961ad6a08d53e09e124b7fff9f3d9d540ff0993e', 'key_dropout_root': '956882526c236f3913d7d02bd60f11b5f39547189b55c38b070863c0f13fd9fe'}
dataset hash present: True


In [12]:
# Concise pipeline summary
merged_size_kb = Path(MERGED_PATH).stat().st_size / 1024
params_size_kb = Path(PARAMS_PATH).stat().st_size / 1024

print("Pipeline summary")
print("=" * 50)
print(f"Generated dataset : {Path(MERGED_PATH).name}  ({merged_size_kb:.0f} KB)")
print(f"  {N_SAMPLES} samples")
print()
print(f"Trained model     : params.msgpack  ({params_size_kb:.0f} KB)")
print(f"  {MODEL_TYPE}  hidden_dim={HIDDEN_DIM}  {NUM_EPOCHS} epochs")
print()
print(f"Optimisation      : {TARGET} (n={NODES})  {OPT_SAMPLES} samples  up to {OPT_STEPS} steps")
print(f"  Best PyTheus fidelity: {best_fid:.4f}")
print(f"  Results in: {result_dir.relative_to(REPO_ROOT)}")

Pipeline summary
Generated dataset : dataset_merged.npz  (79 KB)
  500 samples

Trained model     : params.msgpack  (64 KB)
  PNN  hidden_dim=400  15 epochs

Optimisation      : GHZ (n=4)  3 samples  up to 200 steps
  Best PyTheus fidelity: 0.1255
  Results in: results/sample_workflow/inverse_design/GHZ_n4/GHZ_n4_3


---
## Cleanup

All files generated by this notebook are inside:

```
results/sample_workflow/
```

To remove them completely:

```bash
rm -rf results/sample_workflow/
```

This folder is already listed in `.gitignore` (`results/`) and will not be committed.

---

## Next steps

Once you have confirmed that the notebook runs end-to-end:

1. **Scale up** -- increase `N_SAMPLES`, `NUM_EPOCHS`, and `OPT_STEPS` in the Parameters cell.
2. **Edit the top-of-file constants** in `01_data_generate.py`, `02_ml_model.py`,
   `inverse_design.py` directly for larger standalone runs from the terminal --
   there is no separate config file to keep in sync.
3. **Use a GPU** -- JAX detects and uses a GPU automatically if one is available.
4. **Quick-test script** -- `python notebooks/run_quick_test.py` verifies the
   full pipeline in ~15s without opening any notebook.